In [2]:
import numpy as np
import rasterio
import pandas as pd
from pathlib import Path
import os

# Paths
NAS_ORTHO = Path(r"\\kgkn-nas\eo_data_2\GURS_podatki\DOF\DOF025")
MASK_DIR   = Path(r"C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\processed\road_labels\masks")
TILES_DIR  = Path(r"\\kgkn-nas\eo_data_2\GURS_podatki\DOF\DOF025_tiles")
META_PATH  = Path(r"C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\processed\metadata\selected_orthophotos.csv")
TILE_INDEX = Path(r"C:\Users\lstojano\Desktop\teza\road_extraction_slovenia\data\processed\metadata\tile_index.csv")

# Parameters
TILE_SIZE  = 512
TILE_STEP  = 384   # 25% overlap

# Load selected orthophotos
selected = pd.read_csv(META_PATH)

# Deduplicate by tile_name (same tile can appear in multiple municipalities)
selected = selected.drop_duplicates(subset='tile_name').reset_index(drop=True)

print(f"Unique orthophoto tiles: {len(selected)}")
print(f"Tile size: {TILE_SIZE}x{TILE_SIZE} px")
print(f"Step size: {TILE_STEP} px ({100*(1-TILE_STEP/TILE_SIZE):.0f}% overlap)")

# Estimate tiles per orthophoto
import math
tiles_x = math.ceil((9000 - TILE_SIZE) / TILE_STEP) + 1
tiles_y = math.ceil((12000 - TILE_SIZE) / TILE_STEP) + 1
print(f"\nTiles per orthophoto: {tiles_x} x {tiles_y} = {tiles_x*tiles_y}")
print(f"Max total tiles: {len(selected) * tiles_x * tiles_y:,}")

Unique orthophoto tiles: 1025
Tile size: 512x512 px
Step size: 384 px (25% overlap)

Tiles per orthophoto: 24 x 31 = 744
Max total tiles: 762,600


In [ ]:
import time

records = []
errors = []
processed = 0
skipped = 0
total = len(selected)

print(f"Starting tile generation for {total} orthophotos...")
print("Progress every 50 orthophotos\n")

start_time = time.time()

for idx, row in selected.iterrows():
    
    # Build paths
    tif_path = Path(row['tif_path'])
    mask_path = MASK_DIR / f"{row['tile_name']}_mask.tif"
    split = row['split']
    out_dir = TILES_DIR / split
    
    # Check files exist
    if not tif_path.exists():
        errors.append({'tile': row['tile_name'], 'error': 'TIF not found'})
        continue
    if not mask_path.exists():
        errors.append({'tile': row['tile_name'], 'error': 'mask not found'})
        continue
    
    try:
        # Read orthophoto
        with rasterio.open(tif_path) as src:
            img = src.read()  # shape: (3, H, W)
            img = np.moveaxis(img, 0, -1)  # → (H, W, 3)
        
        # Read mask
        with rasterio.open(mask_path) as src:
            mask = src.read(1)  # shape: (H, W)
        
        H, W = mask.shape
        
        # Slide window
        for y in range(0, H - TILE_SIZE + 1, TILE_STEP):
            for x in range(0, W - TILE_SIZE + 1, TILE_STEP):
                
                img_tile  = img[y:y+TILE_SIZE, x:x+TILE_SIZE, :]
                mask_tile = mask[y:y+TILE_SIZE, x:x+TILE_SIZE]
                
                # Calculate class ratios
                total_px = TILE_SIZE * TILE_SIZE
                bg   = np.sum(mask_tile == 0) / total_px
                c1   = np.sum(mask_tile == 1) / total_px
                c2   = np.sum(mask_tile == 2) / total_px
                c3   = np.sum(mask_tile == 3) / total_px
                road = c1 + c2 + c3
                
                # Generate tile ID
                tile_id = f"{row['tile_name']}_y{y:05d}_x{x:05d}"
                
                # Save paths
                img_save  = out_dir / f"{tile_id}_img.npy"
                mask_save = out_dir / f"{tile_id}_mask.npy"
                
                # Skip if already exists
                if img_save.exists():
                    skipped += 1
                    continue
                
                # Save tiles
                np.save(img_save,  img_tile)
                np.save(mask_save, mask_tile)
                
                # Record metadata
                records.append({
                    'tile_id':          tile_id,
                    'image_path':       str(img_save),
                    'mask_path':        str(mask_save),
                    'source_ortho':     row['tile_name'],
                    'municipality':     row['municipality'],
                    'split':            split,
                    'landscape_type':   row['landscape_type'],
                    'road_pixel_ratio': round(road, 4),
                    'class_1_ratio':    round(c1,   4),
                    'class_2_ratio':    round(c2,   4),
                    'class_3_ratio':    round(c3,   4),
                    'background_ratio': round(bg,   4),
                    'y_offset':         y,
                    'x_offset':         x
                })
        
        processed += 1
        if processed % 50 == 0:
            elapsed = time.time() - start_time
            rate = processed / elapsed
            remaining = (total - processed) / rate / 3600
            print(f"  {processed}/{total} orthos done | "
                  f"{len(records):,} tiles | "
                  f"{rate:.1f} orthos/s | "
                  f"~{remaining:.1f}h remaining")
    
    except Exception as e:
        errors.append({'tile': row['tile_name'], 'error': str(e)})

# Save metadata
tile_df = pd.DataFrame(records)
TILE_INDEX.parent.mkdir(parents=True, exist_ok=True)
tile_df.to_csv(TILE_INDEX, index=False)

elapsed_total = time.time() - start_time
print(f"\n=== Tile generation complete ===")
print(f"Orthophotos processed: {processed}")
print(f"Tiles generated:       {len(records):,}")
print(f"Tiles skipped:         {skipped:,}")
print(f"Errors:                {len(errors)}")
print(f"Total time:            {elapsed_total/3600:.2f}h")
print(f"\nTile index saved to: {TILE_INDEX}")
if errors:
    print("\nFirst 5 errors:")
    for e in errors[:5]:
        print(f"  {e['tile']}: {e['error']}")

Starting tile generation for 1025 orthophotos...
Progress every 50 orthophotos

